In [11]:
!pip install optuna mlflow dagshub seaborn lightgbm

In [12]:
import pandas as pd
import numpy as np
import optuna
import json
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

import lightgbm as lgb
import dagshub
import mlflow
import mlflow.lightgbm

In [13]:
dagshub.init(
    repo_owner="Aryanupadhyay23",
    repo_name="Twitter-Sentiment-Analysis-MLOps",
    mlflow=True
)

mlflow.set_tracking_uri(
    "https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow"
)

mlflow.set_experiment("hyperparameter tuning")

Initialized MLflow to track repo "Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps"

Repository Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps initialized!

<Experiment: artifact_location='mlflow-artifacts:/e8aa851b064b48618f790743afea7af8', creation_time=1771822080177, experiment_id='6', last_update_time=1771822080177, lifecycle_stage='active', name='hyperparameter tuning', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [14]:
df = pd.read_csv("/kaggle/input/datasets/aryanumri/twitter-sentiment/twitter_cleaned (2).csv")

df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)

label_encoder = LabelEncoder()
df["sentiment_encoded"] = label_encoder.fit_transform(df["sentiment"])

print("Classes:", label_encoder.classes_)

Classes: ['negative' 'neutral' 'positive']


In [15]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"],
    df["sentiment_encoded"],
    test_size=0.2,
    stratify=df["sentiment_encoded"],
    random_state=42
)

y_train = np.array(y_train)
y_test = np.array(y_test)

In [16]:
vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    max_features=8000,
    min_df=2
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

num_classes = len(np.unique(y_train))

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Num classes:", num_classes)

Train shape: (41292, 8000)
Test shape: (10323, 8000)
Num classes: 3


In [17]:
def build_lgb(trial):

    return lgb.LGBMClassifier(
        n_estimators=trial.suggest_int("n_estimators", 200, 800),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3),
        max_depth=trial.suggest_int("max_depth", -1, 15),
        num_leaves=trial.suggest_int("num_leaves", 20, 200),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 50),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        objective="multiclass",
        num_class=num_classes,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

In [18]:
def objective(trial):

    model = build_lgb(trial)

    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}"):

        model.fit(
            X_train,
            y_train,
            callbacks=[lgb.log_evaluation(0)]  # disables training logs
        )

        preds = model.predict(X_test)

        macro = f1_score(y_test, preds, average="macro")
        weighted = f1_score(y_test, preds, average="weighted")
        acc = accuracy_score(y_test, preds)

        mlflow.log_params(trial.params)
        mlflow.log_metric("macro_f1", macro)
        mlflow.log_metric("weighted_f1", weighted)
        mlflow.log_metric("accuracy", acc)

    return macro

In [19]:
study = optuna.create_study(
    direction="maximize",
    study_name="lightgbm_study",
    storage="sqlite:///lightgbm_optuna.db",
    load_if_exists=True
)

[I 2026-02-23 08:44:35,726] A new study created in RDB with name: lightgbm_study


In [20]:
with mlflow.start_run(run_name="LightGBM"):

    mlflow.log_param("model_name", "LightGBM")
    mlflow.log_param("vectorizer", "CountVectorizer")
    mlflow.log_param("ngram_range", "(1,2)")
    mlflow.log_param("max_features", 8000)
    mlflow.log_param("min_df", 2)
    mlflow.log_param("train_samples", X_train.shape[0])
    mlflow.log_param("num_features", X_train.shape[1])

    # Hyperparameter tuning
    study.optimize(objective, n_trials=50)

    best_params = study.best_params

    print("Best Macro F1:", study.best_value)
    print("Best Params:", best_params)

    mlflow.log_params(best_params)
    mlflow.log_metric("best_single_split_macro_f1", study.best_value)

    # Train final model
    final_model = lgb.LGBMClassifier(
        **best_params,
        objective="multiclass",
        num_class=num_classes,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    final_model.fit(
        X_train,
        y_train,
        callbacks=[lgb.log_evaluation(0)]
    )

    # 5-Fold CV
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    cv_macro = []
    cv_acc = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):

        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model = lgb.LGBMClassifier(
            **best_params,
            objective="multiclass",
            num_class=num_classes,
            random_state=42,
            n_jobs=-1,
            verbosity=-1
        )

        model.fit(
            X_tr,
            y_tr,
            callbacks=[lgb.log_evaluation(0)]
        )

        preds = model.predict(X_val)

        macro = f1_score(y_val, preds, average="macro")
        acc = accuracy_score(y_val, preds)

        cv_macro.append(macro)
        cv_acc.append(acc)

        print(f"Fold {fold} - Macro F1: {macro:.4f}")

    mlflow.log_metric("cv_macro_f1_mean", np.mean(cv_macro))
    mlflow.log_metric("cv_macro_f1_std", np.std(cv_macro))
    mlflow.log_metric("cv_accuracy_mean", np.mean(cv_acc))

    # Final test evaluation
    preds_test = final_model.predict(X_test)

    test_macro = f1_score(y_test, preds_test, average="macro")
    test_weighted = f1_score(y_test, preds_test, average="weighted")
    test_accuracy = accuracy_score(y_test, preds_test)

    mlflow.log_metric("test_macro_f1", test_macro)
    mlflow.log_metric("test_weighted_f1", test_weighted)
    mlflow.log_metric("test_accuracy", test_accuracy)

    print("Final Test Macro F1:", test_macro)

    # Artifacts
    report = classification_report(y_test, preds_test, output_dict=True)
    with open("classification_report_lgb.json", "w") as f:
        json.dump(report, f, indent=4)
    mlflow.log_artifact("classification_report_lgb.json")

    cm = confusion_matrix(y_test, preds_test)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix - LightGBM")
    plt.savefig("confusion_matrix_lgb.png")
    plt.close()
    mlflow.log_artifact("confusion_matrix_lgb.png")

    study.trials_dataframe().to_csv("optuna_trials_lgb.csv", index=False)
    mlflow.log_artifact("optuna_trials_lgb.csv")

    mlflow.lightgbm.log_model(final_model, artifact_path="model")

print("LightGBM experiment completed successfully.")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-02-23 08:44:48,013] Trial 0 finished with value: 0.7919286765189261 and parameters: {'n_estimators': 800, 'learning_rate': 0.24344498369845224, 'max_depth': 10, 'num_leaves': 111, 'min_child_samples': 35, 'subsample': 0.807684866113124, 'colsample_bytree': 0.7862735411193856}. Best is trial 0 with value: 0.7919286765189261.


🏃 View run trial_0 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/d47554ec0cec4c4e89dc8d54a9b8edda
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-02-23 08:44:57,347] Trial 1 finished with value: 0.746565858548739 and parameters: {'n_estimators': 232, 'learning_rate': 0.14839141728624908, 'max_depth': 8, 'num_leaves': 67, 'min_child_samples': 11, 'subsample': 0.6607682262064434, 'colsample_bytree': 0.8723745314828167}. Best is trial 0 with value: 0.7919286765189261.


🏃 View run trial_1 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/3be9533b0b06430cbf696bde2966d16c
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-02-23 08:45:03,054] Trial 2 finished with value: 0.6979529229199102 and parameters: {'n_estimators': 659, 'learning_rate': 0.2222263403760706, 'max_depth': 3, 'num_leaves': 130, 'min_child_samples': 45, 'subsample': 0.7940997201840245, 'colsample_bytree': 0.6442735835434285}. Best is trial 0 with value: 0.7919286765189261.


...

Fold 5 - Macro F1: 0.8898


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Final Test Macro F1: 0.906384651396785


2026/02/23 09:10:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/23 09:10:47 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LightGBM at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/821036ab8da7475483b8efef7498876f
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6
LightGBM experiment completed successfully.
